# Optional Local Models, Hidden States, and Adaptation

**Optional extension · 25–40 minutes**

This notebook separates four engineering choices:

- **Transformers:** best for teaching logits and hidden states;
- **Ollama:** easiest lightweight local serving path;
- **Unsloth:** optional Colab GPU route for LoRA/SFT;
- **vLLM:** instructor/server route for high-throughput GPU inference.

All model downloads and training cells default to off. The notebook runs end-to-end on an ordinary laptop without downloading a model.

In [1]:
from pathlib import Path
import json
import os
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 3000
rng = np.random.default_rng(SEED)
pd.set_option("display.max_colwidth", 120)
plt.style.use("seaborn-v0_8-whitegrid")

In [2]:
def find_data_file():
    candidates = [
        Path("data/c13k_tutorial_sample.csv"),
        Path("c13k_tutorial_sample.csv"),
        Path("/content/data/c13k_tutorial_sample.csv"),
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(
        "Could not find c13k_tutorial_sample.csv. Keep the data/ folder next to the notebook, "
        "or upload the CSV to Colab."
    )


DATA_PATH = find_data_file()
data = pd.read_csv(DATA_PATH)

# This tutorial predicts a binary majority label. Exact 50/50 rows have no majority.
data = data.loc[data["bRate"] != 0.5].copy()
data["majority_B"] = (data["bRate"] > 0.5).astype(int)
data["ev_diff"] = data["ev_B"] - data["ev_A"]
data["risk_diff"] = data["sd_B"] - data["sd_A"]
data["worst_diff"] = data["min_B"] - data["min_A"]
data["best_diff"] = data["max_B"] - data["max_A"]

print(f"Loaded {len(data):,} real Choice13K condition rows from {DATA_PATH}")
print(f"Unique problem IDs: {data['Problem'].nunique():,}")
data.head(3)

Loaded 1,926 real Choice13K condition rows from data\c13k_tutorial_sample.csv
Unique problem IDs: 1,926


,row_id,Problem,Feedback,n,Block,Ha,pHa,La,Hb,pHb,...,max_A,max_B,n_outcomes_A,n_outcomes_B,majority_choice,majority_B,ev_diff,risk_diff,worst_diff,best_diff
0,4,5,False,15,1,26,1.0,26,45,0.75,...,26.0,71.0,1,6,B,1,-1.25,35.688759,-62.0,45.0
1,6,7,False,15,1,-6,1.0,-6,24,0.20,...,-6.0,24.5,1,3,B,1,6.80,11.602155,1.0,30.5
2,8,8,False,15,1,27,1.0,27,89,0.50,...,27.0,89.0,1,2,A,0,5.50,56.500000,-51.0,62.0


In [3]:
SYSTEM_PROMPT = (
    "You estimate aggregate human behavior in a risky-choice experiment. "
    "Do not choose for yourself."
)
LABEL_SYSTEM_PROMPT = (
    "You predict the human majority label in a risky-choice experiment. "
    "Respond with exactly one label: A or B."
)


def format_trial(row, include_answer=False):
    feedback = (
        "Outcome feedback is shown after each choice."
        if bool(row["Feedback"])
        else "No outcome feedback is shown."
    )
    text = (
        f"Option A: {row['lottery_A']}\n"
        f"Option B: {row['lottery_B']}\n"
        f"Condition: {feedback} Block {int(row['Block'])}."
    )
    if include_answer:
        text += f"\nObserved aggregate p_B: {row['bRate']:.3f}"
    return text


def zero_shot_messages(row):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": (
                format_trial(row)
                + "\nParticipants faced this pair five times. Predict the average fraction "
                "of decisions allocated to Option B. Return JSON only: "
                '{"p_B": number between 0 and 1}.'
            ),
        },
    ]


def label_messages(row):
    return [
        {"role": "system", "content": LABEL_SYSTEM_PROMPT},
        {"role": "user", "content": format_trial(row) + "\nAnswer:"},
    ]

## 1. Direct logits with Transformers

Recommended teaching model: `Qwen/Qwen3-0.6B`.

- small enough for a short Colab demonstration;
- CPU is possible but slower;
- use a pre-downloaded local directory when classroom network access is uncertain.

For a fresh environment:

```python
%pip install -q "transformers>=4.51" accelerate torch
```

In [4]:
RUN_TRANSFORMERS = False
MODEL_ID = "Qwen/Qwen3-0.6B"
LOCAL_MODEL_DIR = None   # e.g., "/content/models/Qwen3-0.6B"
MODEL_SOURCE = LOCAL_MODEL_DIR or MODEL_ID

if RUN_TRANSFORMERS:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM

    tokenizer = AutoTokenizer.from_pretrained(MODEL_SOURCE)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_SOURCE,
        torch_dtype="auto",
        device_map="auto",
    )
    model.eval()
    print("Loaded", MODEL_SOURCE)
else:
    print("Skipped model download. Set RUN_TRANSFORMERS=True to run the local lab.")

Skipped model download. Set RUN_TRANSFORMERS=True to run the local lab.


### Candidate-sequence scoring

Top-k output is convenient, but a valid label may be absent or may tokenize into multiple pieces. The function below scores every token in candidate completions `A` and `B`, then normalizes their sequence scores.

In [5]:
local_trial = data.loc[~data["Amb"].astype(bool)].sample(1, random_state=SEED).iloc[0]
plain_prompt = (
    LABEL_SYSTEM_PROMPT + "\n\n" + format_trial(local_trial)
    + "\nRespond with exactly one letter.\nAnswer:"
)
print(plain_prompt)

You predict the human majority label in a risky-choice experiment. Respond with exactly one label: A or B.

Option A: $22 for sure
Option B: 1% chance of $8; 49.5% chance of $12; 24.75% chance of $14; 12.375% chance of $18; 6.188% chance of $26; 3.094% chance of $42; 1.547% chance of $74; 1.547% chance of $138
Condition: No outcome feedback is shown. Block 1.
Respond with exactly one letter.
Answer:


In [6]:
def candidate_sequence_logprob(model, tokenizer, prompt, candidate):
    """Sum conditional log probabilities for every token in a candidate string."""
    import torch

    prompt_ids = tokenizer(prompt, add_special_tokens=True)["input_ids"]
    candidate_ids = tokenizer(candidate, add_special_tokens=False)["input_ids"]
    input_ids = torch.tensor(
        [prompt_ids + candidate_ids], device=model.device, dtype=torch.long
    )
    with torch.no_grad():
        logits = model(input_ids=input_ids).logits[0]
    log_probs = torch.log_softmax(logits, dim=-1)

    start = len(prompt_ids) - 1
    score = 0.0
    for offset, token_id in enumerate(candidate_ids):
        score += float(log_probs[start + offset, token_id].cpu())
    return score, candidate_ids


def score_a_vs_b(model, tokenizer, prompt):
    candidates = ["A", "B"]
    scored = {
        label: candidate_sequence_logprob(model, tokenizer, prompt, label)
        for label in candidates
    }
    log_scores = np.array([scored[label][0] for label in candidates])
    probabilities = np.exp(log_scores - log_scores.max())
    probabilities /= probabilities.sum()
    return pd.DataFrame({
        "label": candidates,
        "token_ids": [scored[label][1] for label in candidates],
        "sequence_logprob": log_scores,
        "renormalized_probability": probabilities,
    })


if RUN_TRANSFORMERS:
    display(score_a_vs_b(model, tokenizer, plain_prompt))
else:
    print("Function defined; execution skipped.")

Function defined; execution skipped.


### Inspect the actual top tokens

Raw logits become log probabilities after `log_softmax`. Looking at the top tokens before restricting to A/B makes the re-normalization step explicit.

In [7]:
if RUN_TRANSFORMERS:
    import torch
    encoded = tokenizer(plain_prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model(**encoded)
    final_logits = output.logits[0, -1]
    final_logprobs = torch.log_softmax(final_logits, dim=-1)
    values, token_ids = torch.topk(final_logprobs, k=8)
    top_tokens = pd.DataFrame({
        "token": [tokenizer.decode([int(i)]) for i in token_ids],
        "token_id": token_ids.cpu().numpy(),
        "logprob": values.cpu().numpy(),
        "probability": values.exp().cpu().numpy(),
    })
    display(top_tokens)
else:
    print("Skipped forward pass.")

Skipped forward pass.


## 2. Hidden states: a representation, not a mechanism

The next cell extracts the final-token hidden state from selected layers. A downstream probe can test whether these vectors carry EV, risk, or choice information. Similar geometry across model and brain data is evidence of correspondence—not proof of an identical computation.

In [8]:
def extract_layer_vectors(model, tokenizer, prompts, layers=(-1, -7, -14)):
    import torch
    vectors = {layer: [] for layer in layers}
    for prompt in prompts:
        encoded = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            output = model(**encoded, output_hidden_states=True)
        for layer in layers:
            vector = output.hidden_states[layer][0, -1].float().cpu().numpy()
            vectors[layer].append(vector)
    return {layer: np.vstack(items) for layer, items in vectors.items()}


if RUN_TRANSFORMERS:
    representation_rows = data.loc[~data["Amb"].astype(bool)].sample(12, random_state=SEED)
    prompts = [SYSTEM_PROMPT + "\n" + format_trial(row) for _, row in representation_rows.iterrows()]
    layer_vectors = extract_layer_vectors(model, tokenizer, prompts)
    print({layer: matrix.shape for layer, matrix in layer_vectors.items()})
else:
    print("Hidden-state extraction defined; execution skipped.")

Hidden-state extraction defined; execution skipped.


## 3. Ollama: the easiest local serving path

Install Ollama separately, then pull a small quantized model before class:

```bash
ollama pull qwen3:0.6b
```

Ollama exposes an OpenAI-compatible endpoint, so only the base URL and model change. It is excellent for local privacy and deployment demos, but it does not expose layer-wise hidden states.

In [9]:
RUN_OLLAMA = False

if RUN_OLLAMA:
    from openai import OpenAI
    ollama_client = OpenAI(api_key="ollama", base_url="http://localhost:11434/v1")
    response = ollama_client.chat.completions.create(
        model="qwen3:0.6b",
        messages=zero_shot_messages(local_trial),
        temperature=0,
        max_tokens=8,
    )
    print(response.choices[0].message.content)
else:
    print("Skipped. Start Ollama locally and set RUN_OLLAMA=True.")

Skipped. Start Ollama locally and set RUN_OLLAMA=True.


## 4. Prepare SFT data before training anything

Supervised fine-tuning learns to imitate target outputs. Here the target is the *aggregate B-choice rate*, so improved performance would demonstrate behavioral imitation—not mechanistic recovery.

Always split by problem ID before constructing training examples.

In [10]:
from sklearn.model_selection import GroupShuffleSplit

sft_source = data.loc[~data["Amb"].astype(bool)].copy()
split = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
train_idx, validation_idx = next(
    split.split(sft_source, groups=sft_source["Problem"])
)
sft_train = sft_source.iloc[train_idx]
sft_validation = sft_source.iloc[validation_idx]
assert set(sft_train["Problem"]).isdisjoint(sft_validation["Problem"])


def to_chat_example(row):
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": format_trial(row) + "\nAnswer:"},
            {"role": "assistant", "content": json.dumps({"p_B": round(float(row["bRate"]), 3)})},
        ],
        "metadata": {
            "problem_id": int(row["Problem"]),
            "human_bRate": float(row["bRate"]),
            "n": int(row["n"]),
        },
    }


sft_examples = [to_chat_example(row) for _, row in sft_train.head(5).iterrows()]
print(json.dumps(sft_examples[0], indent=2))

{
  "messages": [
    {
      "role": "system",
      "content": "You estimate aggregate human behavior in a risky-choice experiment. Do not choose for yourself."
    },
    {
      "role": "user",
      "content": "Option A: $26 for sure\nOption B: 25% chance of -$36; 37.5% chance of $41; 18.75% chance of $43; 9.375% chance of $47; 4.688% chance of $55; 4.688% chance of $71\nCondition: No outcome feedback is shown. Block 1.\nAnswer:"
    },
    {
      "role": "assistant",
      "content": "{\"p_B\": 0.587}"
    }
  ],
  "metadata": {
    "problem_id": 5,
    "human_bRate": 0.5866666666666667,
    "n": 15
  }
}


<!-- RESTORED_WORKFLOW: lora_training -->
### Optional LoRA training and held-out evaluation

This is real training code, not pseudocode. It supervises only the final
A/B answer token, updates LoRA parameters on grouped training problems, and
evaluates probability estimates on held-out problem IDs. It is disabled by
default because it requires a CUDA GPU and downloads the local checkpoint.

In [11]:
# RESTORED_WORKFLOW: lora_training
RUN_LORA_SFT = False

if RUN_LORA_SFT:
    if not RUN_TRANSFORMERS:
        raise RuntimeError("Set RUN_TRANSFORMERS=True and rerun the model-loading cells first.")
    import torch
    from torch.utils.data import DataLoader, Dataset
    from peft import LoraConfig, TaskType, get_peft_model

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    def encode_supervised_example(row):
        prompt = LABEL_SYSTEM_PROMPT + "\n\n" + format_trial(row) + "\nAnswer:"
        prompt_ids = tokenizer(prompt, add_special_tokens=True)["input_ids"]
        answer = "B" if int(row["majority_B"]) == 1 else "A"
        answer_ids = tokenizer(answer, add_special_tokens=False)["input_ids"]
        return {
            "input_ids": prompt_ids + answer_ids,
            "labels": [-100] * len(prompt_ids) + answer_ids,
        }

    class ChoiceDataset(Dataset):
        def __init__(self, frame, limit=None):
            rows = frame.head(limit) if limit else frame
            self.examples = [encode_supervised_example(row) for _, row in rows.iterrows()]
        def __len__(self): return len(self.examples)
        def __getitem__(self, index): return self.examples[index]

    def collate(examples):
        width = max(len(item["input_ids"]) for item in examples)
        input_ids = torch.full((len(examples), width), tokenizer.pad_token_id, dtype=torch.long)
        attention_mask = torch.zeros_like(input_ids)
        labels = torch.full_like(input_ids, -100)
        for row, item in enumerate(examples):
            n = len(item["input_ids"])
            input_ids[row, :n] = torch.tensor(item["input_ids"])
            attention_mask[row, :n] = 1
            labels[row, :n] = torch.tensor(item["labels"])
        return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

    model = get_peft_model(
        model,
        LoraConfig(
            task_type=TaskType.CAUSAL_LM,
            r=8,
            lora_alpha=16,
            lora_dropout=0.05,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        ),
    )
    model.train()
    loader = DataLoader(ChoiceDataset(sft_train, limit=256), batch_size=2, shuffle=True, collate_fn=collate)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)
    training_loss = []
    for epoch in range(2):
        for batch in loader:
            batch = {key: value.to(model.device) for key, value in batch.items()}
            optimizer.zero_grad(set_to_none=True)
            loss = model(**batch).loss
            loss.backward()
            optimizer.step()
            training_loss.append(float(loss.detach().cpu()))
        print(f"epoch {epoch + 1}: mean loss={np.mean(training_loss[-len(loader):]):.4f}")

    # Reuse the candidate-sequence scorer defined above on held-out problem IDs.
    model.eval()
    evaluation_rows = []
    for _, row in sft_validation.head(64).iterrows():
        prompt = LABEL_SYSTEM_PROMPT + "\n\n" + format_trial(row) + "\nAnswer:"
        scores = score_a_vs_b(model, tokenizer, prompt).set_index("label")
        evaluation_rows.append({
            "problem": int(row["Problem"]),
            "human_p_B": float(row["bRate"]),
            "model_p_B": float(scores.loc["B", "renormalized_probability"]),
        })
    lora_evaluation = pd.DataFrame(evaluation_rows)
    display(lora_evaluation.head())
    print("Held-out log loss against majority choice:", float(np.mean(
        -((lora_evaluation["human_p_B"] >= 0.5) * np.log(lora_evaluation["model_p_B"].clip(1e-6, 1-1e-6))
          + (lora_evaluation["human_p_B"] < 0.5) * np.log((1-lora_evaluation["model_p_B"]).clip(1e-6, 1-1e-6)))
    )))
else:
    print("LoRA training is off; set RUN_TRANSFORMERS=True and RUN_LORA_SFT=True on a GPU runtime.")

LoRA training is off; set RUN_TRANSFORMERS=True and RUN_LORA_SFT=True on a GPU runtime.


## 5. Where RL and vLLM belong

**RL / preference optimization** is optional because the reward definition is the scientific commitment. Rewarding agreement with aggregate choice rates may improve imitation while reinforcing a shortcut. Show the reward and its failure modes before showing an optimizer.

**vLLM** becomes useful when one GPU server must support many concurrent students or a large batch. It is not the simplest per-student laptop setup, and native Windows is not its main deployment path.

## Recommended teaching stack

| Need | Default |
|---|---|
| Cloud generation / JSON in China | GLM via OpenAI-compatible client |
| Token logprobs | DeepSeek non-thinking mode |
| Local generation on a laptop | Ollama |
| Raw logits and hidden states | Transformers |
| Optional Colab LoRA/SFT | Unsloth |
| Shared high-throughput GPU service | vLLM |

The simplest path is intentionally not one package for everything: **Transformers teaches research signals; Ollama teaches deployment.**